In [1]:
import objaverse
import objaverse.xl as oxl

import pandas as pd
import numpy as np
import random

# Meshes

In [2]:
obj_path = "/vol/isy-rl/dtrofimov/data/objaverse"

In [3]:
annotations = oxl.get_annotations(
    download_dir=obj_path
)

In [4]:
annotations["metadata"] = annotations["metadata"].apply(eval)

In [5]:
annotations[annotations["metadata"].apply(lambda x: len(x) > 0)]

,fileIdentifier,source,license,fileType,sha256,metadata
5236361,https://thingiverse.com/thing:2609614/files?fi...,thingiverse,Creative Commons - Attribution - Non-Commercial,stl,82e4d7eb817c1a352d7a12def0b78b66991fb71efdfa08...,{'filename': 'Roof_bottom.stl'}
5236362,https://thingiverse.com/thing:2609614/files?fi...,thingiverse,Creative Commons - Attribution - Non-Commercial,stl,fd389685afabfdbfaf3486c34ea73d1363c7ce7acac9df...,{'filename': 'Stable_base.stl'}
5236363,https://thingiverse.com/thing:2609614/files?fi...,thingiverse,Creative Commons - Attribution - Non-Commercial,stl,6dabb0fbf32d6093207cf6f128b659c817882f43ab3f4c...,{'filename': 'Roof_left.stl'}
5236364,https://thingiverse.com/thing:2609614/files?fi...,thingiverse,Creative Commons - Attribution - Non-Commercial,stl,147f69858bee722bb96a4094c28da5f5f6c0924f023939...,{'filename': 'Roof_right.stl'}
5236365,https://thingiverse.com/thing:3522959/files?fi...,thingiverse,Creative Commons - Attribution - Non-Commercial,stl,0fd4c8e534d12cb484694f8c5f91043fcef845809d7f8c...,{'filename': 'Pictureframe_Wallmount.stl'}
...,...,...,...,...,...,...
8970975,https://3d-api.si.edu/content/document/3d_pack...,smithsonian,Creative Commons Zero v1.0 Universal,glb,4b6b07001626cfc280249df36590047c3364e7a779c7f1...,{'title': 'Prototype of a digital heart rhythm...
8970976,https://3d-api.si.edu/content/document/3d_pack...,smithsonian,Creative Commons Zero v1.0 Universal,glb,2c9fbe4e671def77f08c50b64b5668e41e00a6f7d54397...,{'title': 'Red Starfleet uniform worn by Niche...
8970977,https://3d-api.si.edu/content/document/3d_pack...,smithsonian,Creative Commons Zero v1.0 Universal,glb,0547338bfa42ad3fa4b7668ae88c468653a91de7b8bc56...,{'title': 'Flight suit worn by Charles F. Bold...
8970978,https://3d-api.si.edu/content/document/3d_pack...,smithsonian,Creative Commons Zero v1.0 Universal,glb,f69b62bb07a464ee4bbdfaa8f66bfd7ff4d2281d7f9bc6...,{'title': 'Costume worn by Nona Hendryx of Lab...


In [6]:
annotations["keys"] = annotations["metadata"].apply(lambda info: list(info.keys()))

In [7]:
annotations[["keys"]].explode("keys")["keys"].unique()

array([nan, 'filename', 'title'], dtype=object)

In [8]:
uids = objaverse.load_uids()

In [9]:
len(uids)

798759

In [ ]:
#It breaks since it tries to download everything into home dir
objaverse.load_annotations()

# Loading into another dir

In [10]:
import glob
import gzip
import json
import multiprocessing
import os
import urllib.request
import warnings
from typing import Any, Dict, List, Optional, Tuple

from tqdm import tqdm

BASE_PATH = "/vol/isy-rl/dtrofimov/data/objaverse"#os.path.join(os.path.expanduser("~"), ".objaverse")

__version__ = "<REPLACE_WITH_VERSION>"
_VERSIONED_PATH = os.path.join(BASE_PATH, "hf-objaverse-v1")


def load_annotations(uids: Optional[List[str]] = None) -> Dict[str, Any]:
    """Load the full metadata of all objects in the dataset.

    Args:
        uids: A list of uids with which to load metadata. If None, it loads
        the metadata for all uids.

    Returns:
        A dictionary mapping the uid to the metadata.
    """
    metadata_path = os.path.join(_VERSIONED_PATH, "metadata")
    object_paths = _load_object_paths()
    dir_ids = (
        set(object_paths[uid].split("/")[1] for uid in uids)
        if uids is not None
        else [f"{i // 1000:03d}-{i % 1000:03d}" for i in range(160)]
    )
    if len(dir_ids) > 10:
        dir_ids = tqdm(dir_ids)
    out = {}
    for i_id in dir_ids:
        json_file = f"{i_id}.json.gz"
        local_path = os.path.join(metadata_path, json_file)
        if not os.path.exists(local_path):
            hf_url = f"https://huggingface.co/datasets/allenai/objaverse/resolve/main/metadata/{i_id}.json.gz"
            # wget the file and put it in local_path
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            urllib.request.urlretrieve(hf_url, local_path)
        with gzip.open(local_path, "rb") as f:
            data = json.load(f)
        if uids is not None:
            data = {uid: data[uid] for uid in uids if uid in data}
        out.update(data)
        if uids is not None and len(out) == len(uids):
            break
    return out

def _load_object_paths() -> Dict[str, str]:
    """Load the object paths from the dataset.

    The object paths specify the location of where the object is located
    in the Hugging Face repo.

    Returns:
        A dictionary mapping the uid to the object path.
    """
    object_paths_file = "object-paths.json.gz"
    local_path = os.path.join(_VERSIONED_PATH, object_paths_file)
    if not os.path.exists(local_path):
        hf_url = f"https://huggingface.co/datasets/allenai/objaverse/resolve/main/{object_paths_file}"
        # wget the file and put it in local_path
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        urllib.request.urlretrieve(hf_url, local_path)
    with gzip.open(local_path, "rb") as f:
        object_paths = json.load(f)
    return object_paths

In [11]:
all_annotations = load_annotations(uids)

 99%|██████████████████████████████████████████████████████████▋| 159/160 [01:21<00:00,  1.95it/s]


In [12]:
categories = list(map(lambda v: list(map(lambda x: x["name"], v[1]["categories"])), all_annotations.items()))

In [13]:
len(categories)

798759

In [14]:
categories

[['electronics-gadgets'],
 [],
 [],
 [],
 ['characters-creatures', 'science-technology'],
 ['characters-creatures', 'people'],
 [],
 ['cultural-heritage-history', 'science-technology'],
 [],
 ['characters-creatures'],
 [],
 [],
 [],
 ['animals-pets', 'cars-vehicles'],
 ['nature-plants'],
 [],
 ['electronics-gadgets'],
 [],
 [],
 ['furniture-home'],
 ['cultural-heritage-history', 'nature-plants'],
 ['architecture'],
 ['characters-creatures'],
 ['weapons-military'],
 [],
 ['cultural-heritage-history'],
 [],
 ['cultural-heritage-history', 'weapons-military'],
 [],
 [],
 ['art-abstract', 'furniture-home'],
 [],
 [],
 ['art-abstract', 'characters-creatures'],
 ['electronics-gadgets', 'science-technology'],
 [],
 ['cultural-heritage-history'],
 [],
 [],
 [],
 ['architecture'],
 [],
 ['nature-plants'],
 ['architecture'],
 [],
 ['architecture'],
 ['furniture-home'],
 [],
 ['characters-creatures'],
 [],
 [],
 ['cultural-heritage-history', 'furniture-home'],
 ['architecture', 'news-politics'],
 

In [15]:
from itertools import chain

In [16]:
categories = list(chain(*categories))

In [17]:
df_categ = pd.DataFrame({"categories": categories})

In [18]:
df_categ.groupby("categories", as_index=False).count()

,categories
0,animals-pets
1,architecture
2,art-abstract
3,cars-vehicles
4,characters-creatures
5,cultural-heritage-history
6,electronics-gadgets
7,fashion-style
8,food-drink
9,furniture-home


In [38]:
all_annotations.popitem()

('a7ba6418fee74f83b7bb72d09a3763ff',
 {'uri': 'https://api.sketchfab.com/v3/models/a7ba6418fee74f83b7bb72d09a3763ff',
  'uid': 'a7ba6418fee74f83b7bb72d09a3763ff',
  'name': 'Elote',
  'staffpickedAt': None,
  'viewCount': 115,
  'likeCount': 1,
  'animationCount': 0,
  'viewerUrl': 'https://sketchfab.com/3d-models/a7ba6418fee74f83b7bb72d09a3763ff',
  'embedUrl': 'https://sketchfab.com/models/a7ba6418fee74f83b7bb72d09a3763ff/embed',
  'commentCount': 0,
  'isDownloadable': True,
  'publishedAt': '2019-03-22T08:01:28.076422',
  'tags': [],
  'categories': [{'name': 'characters-creatures'}, {'name': 'nature-plants'}],
  'thumbnails': {'images': [{'uid': 'e229970d3df34cfcb13f6f06e083b1ea',
     'size': 16549,
     'width': 720,
     'url': 'https://media.sketchfab.com/models/a7ba6418fee74f83b7bb72d09a3763ff/thumbnails/eb7ef8aedef248458caf19b2d18746c1/0074ca69629e4fbdae103ecde0c6edc6.jpeg',
     'height': 405},
    {'uid': 'f664d67c21204ca7bb26d3271ad493ee',
     'size': 65460,
     'width'

In [43]:
annotations[annotations["fileIdentifier"] == "https://sketchfab.com/3d-models/a7ba6418fee74f83b7bb72d09a3763ff"]

,fileIdentifier,source,license,fileType,sha256,metadata
9454162,https://sketchfab.com/3d-models/a7ba6418fee74f...,sketchfab,Creative Commons - Attribution,glb,882f8fcf9b4fd3949a34e371a84efa311809fd98d6bd8c...,{}


In [19]:
def filter_ann(annotations: dict, name: str) -> dict:
    return dict(filter(lambda item: name in item[1]["name"].lower(), annotations.items()))

In [20]:
teapots_ann = filter_ann(all_annotations, "teapot")

In [21]:
teapots_ann.popitem()

('af9aa36d163d4b3b895696a0e930095e',
 {'uri': 'https://api.sketchfab.com/v3/models/af9aa36d163d4b3b895696a0e930095e',
  'uid': 'af9aa36d163d4b3b895696a0e930095e',
  'name': 'Tetera de plata / Silver teapot',
  'staffpickedAt': None,
  'viewCount': 35,
  'likeCount': 0,
  'animationCount': 0,
  'viewerUrl': 'https://sketchfab.com/3d-models/af9aa36d163d4b3b895696a0e930095e',
  'embedUrl': 'https://sketchfab.com/models/af9aa36d163d4b3b895696a0e930095e/embed',
  'commentCount': 0,
  'isDownloadable': True,
  'publishedAt': '2021-06-17T03:06:24.786168',
  'tags': [{'name': 'teapot',
    'slug': 'teapot',
    'uri': 'https://api.sketchfab.com/v3/tags/teapot'},
   {'name': 'tete',
    'slug': 'tete',
    'uri': 'https://api.sketchfab.com/v3/tags/tete'},
   {'name': 'silver',
    'slug': 'silver',
    'uri': 'https://api.sketchfab.com/v3/tags/silver'},
   {'name': 'plata',
    'slug': 'plata',
    'uri': 'https://api.sketchfab.com/v3/tags/plata'},
   {'name': 'sterling',
    'slug': 'sterling'

In [22]:
# for the scenes
# TODO add more scenes with easily discovered objects like teapot
objects_to_filter = {
    "poster": ["chair", "poster", "frame", "artwork"],
    "kitchen": ["monitor", "piano", "bulb", "lamp", "table", "toaster", "oven", "fridge", "sink", "plate", "spoon"],
    "stump": ["head", "monitor", "chair", "log", "branch", "bench"],
    "others": ["bookshelf", "bottle", "bowl", "cup", "desk", "door", "keyboard", "lamp", "laptop", "stool",
              "teapot", "microwave", "bed", "sofa", "wardrobe", "rug", "curtains", "mirror", "clock"]
}

In [23]:
def filter_many(annotations: dict, names: list) -> dict:
    result = {}
    for name in names:
        result.update(filter_ann(annotations, name))
    return result

In [24]:
filtered_objects = filter_many(all_annotations, list(chain(*list(objects_to_filter.values()))))

In [85]:
filtered_objects.popitem()

('5af99db17fb645bba7e55ace9e7b6b34',
 {'uri': 'https://api.sketchfab.com/v3/models/5af99db17fb645bba7e55ace9e7b6b34',
  'uid': '5af99db17fb645bba7e55ace9e7b6b34',
  'name': 'Microwave Oven',
  'staffpickedAt': None,
  'viewCount': 3134,
  'likeCount': 69,
  'animationCount': 0,
  'viewerUrl': 'https://sketchfab.com/3d-models/5af99db17fb645bba7e55ace9e7b6b34',
  'embedUrl': 'https://sketchfab.com/models/5af99db17fb645bba7e55ace9e7b6b34/embed',
  'commentCount': 2,
  'isDownloadable': True,
  'publishedAt': '2020-03-26T15:00:11.735732',
  'tags': [{'name': 'microwave',
    'slug': 'microwave',
    'uri': 'https://api.sketchfab.com/v3/tags/microwave'},
   {'name': 'dirty',
    'slug': 'dirty',
    'uri': 'https://api.sketchfab.com/v3/tags/dirty'},
   {'name': 'appliance',
    'slug': 'appliance',
    'uri': 'https://api.sketchfab.com/v3/tags/appliance'},
   {'name': 'old',
    'slug': 'old',
    'uri': 'https://api.sketchfab.com/v3/tags/old'},
   {'name': 'microwave-oven',
    'slug': 'mi

In [89]:
1000 * 10 / 1024

9.765625

In [28]:
filtered_objects['fe07d29f80cf4985a7e0c3e49bcc638c']

{'uri': 'https://api.sketchfab.com/v3/models/fe07d29f80cf4985a7e0c3e49bcc638c',
 'uid': 'fe07d29f80cf4985a7e0c3e49bcc638c',
 'name': 'FNAF Chair',
 'staffpickedAt': None,
 'viewCount': 277,
 'likeCount': 3,
 'animationCount': 0,
 'viewerUrl': 'https://sketchfab.com/3d-models/fe07d29f80cf4985a7e0c3e49bcc638c',
 'embedUrl': 'https://sketchfab.com/models/fe07d29f80cf4985a7e0c3e49bcc638c/embed',
 'commentCount': 0,
 'isDownloadable': True,
 'publishedAt': '2019-05-27T02:17:26.384858',
 'tags': [{'name': 'chairs',
   'slug': 'chairs',
   'uri': 'https://api.sketchfab.com/v3/tags/chairs'},
  {'name': 'fnaf',
   'slug': 'fnaf',
   'uri': 'https://api.sketchfab.com/v3/tags/fnaf'},
  {'name': 'pizzeria',
   'slug': 'pizzeria',
   'uri': 'https://api.sketchfab.com/v3/tags/pizzeria'},
  {'name': 'fnaf1',
   'slug': 'fnaf1',
   'uri': 'https://api.sketchfab.com/v3/tags/fnaf1'},
  {'name': 'fnafmodels',
   'slug': 'fnafmodels',
   'uri': 'https://api.sketchfab.com/v3/tags/fnafmodels'}],
 'categorie

In [38]:
len(filtered_objects)

44478

In [39]:
sampled_objects = np.random.choice(list(map(lambda x: x[1]["viewerUrl"], filtered_objects.items())), 44478, False).tolist()

In [40]:
sampled_objects = {url: True for url in sampled_objects}

In [41]:
sampled_json = dict(filter(lambda item: item[1]["viewerUrl"] in sampled_objects, filtered_objects.items()))

In [42]:
def filter_many_classes(annotations: dict, names: list) -> dict:
    result = {}
    stats = {}
    for name in names:
        filtered = filter_ann(annotations, name)
        result.update(filtered)
        stats[name] = len(filtered)
    return result, stats

In [43]:
r, stats = filter_many_classes(sampled_json, list(chain(*list(objects_to_filter.values()))))

In [46]:
stats

{'chair': 4709,
 'poster': 203,
 'frame': 1305,
 'artwork': 44,
 'monitor': 400,
 'piano': 318,
 'bulb': 273,
 'lamp': 3163,
 'table': 5264,
 'toaster': 97,
 'oven': 360,
 'fridge': 207,
 'sink': 407,
 'plate': 1046,
 'spoon': 182,
 'head': 5563,
 'log': 5304,
 'branch': 146,
 'bench': 1083,
 'bookshelf': 203,
 'bottle': 1648,
 'bowl': 1063,
 'cup': 2187,
 'desk': 1429,
 'door': 2393,
 'keyboard': 327,
 'laptop': 272,
 'stool': 526,
 'teapot': 299,
 'microwave': 90,
 'bed': 2409,
 'sofa': 1276,
 'wardrobe': 174,
 'rug': 435,
 'curtains': 42,
 'mirror': 332,
 'clock': 785}

In [47]:
len(sampled_json)

44478

In [48]:
annotations_sampled = annotations[annotations["fileIdentifier"].isin(sampled_objects)]

In [49]:
annotations_sampled

,fileIdentifier,source,license,fileType,sha256,metadata,keys
8971022,https://sketchfab.com/3d-models/524c5ae4075a4a...,sketchfab,Creative Commons - Attribution,glb,25ada8ac68d82791526979bc062d90ed920882948660df...,{},[]
8971029,https://sketchfab.com/3d-models/6e7fb4a4787241...,sketchfab,Creative Commons - Attribution,glb,5354c846cf5a7f4d900ced80419e3c8311cf72622d697d...,{},[]
8971059,https://sketchfab.com/3d-models/a5d06450f38b43...,sketchfab,Creative Commons - Attribution,glb,19260f69de00dd60836be1f09c511aec16e7a4c93a3126...,{},[]
8971068,https://sketchfab.com/3d-models/96a55276db4b4d...,sketchfab,Creative Commons - Attribution,glb,37609fb61aeb53169cdd917ff86300e045a10b59d2cf3a...,{},[]
8971090,https://sketchfab.com/3d-models/145061eedca04a...,sketchfab,Creative Commons - Attribution,glb,e13c58107fe8d9659704718f74a6a8c86aa95296f3f99f...,{},[]
...,...,...,...,...,...,...,...
9766787,https://sketchfab.com/3d-models/3ea030c2fc5944...,sketchfab,Creative Commons - Attribution,glb,885de304dc9deea850dbd92e2d4b78894c085c0455836f...,{},[]
9766828,https://sketchfab.com/3d-models/117ff7a0393049...,sketchfab,Creative Commons - Attribution,glb,f27c9b92125025d4c9b0bb54c0c7d0d2035f233da570e9...,{},[]
9766873,https://sketchfab.com/3d-models/c1340596c55a45...,sketchfab,Creative Commons - Attribution,glb,e40b7579eb9356f97053631ced8dd4b960edc30fa03709...,{},[]
9766914,https://sketchfab.com/3d-models/808e3269d21446...,sketchfab,Creative Commons - Attribution,glb,97bd5afd4c0d4491d047c78694e60fe25484f7818d812a...,{},[]


In [50]:
objects_to_download = annotations_sampled[["sha256", "fileIdentifier", "source"]].to_dict("records")

In [51]:
import json

In [52]:
with open("object_download_january.json", "w") as f:
    json.dump(objects_to_download, f)

In [124]:
objects_to_download

[{'sha256': '079c1fbedddddc5b9462a22ec9e043e808aac72c0b138b2173b4f191971135c6',
  'fileIdentifier': 'https://sketchfab.com/3d-models/bc0fd04422374b3987162a6c499280d8',
  'source': 'sketchfab'},
 {'sha256': '8a784c4a068239e24666b84acd05083ea7cb5092e3c0bc7550c2e13e5798786f',
  'fileIdentifier': 'https://sketchfab.com/3d-models/2c3379f8126743c0b6e564c70ff859bc',
  'source': 'sketchfab'},
 {'sha256': '4e41726df0b2eb46fa66be100b6c9389c2fb215fa87e17fc818cbea1d920c34d',
  'fileIdentifier': 'https://sketchfab.com/3d-models/007ab9e6f1c047b287bc0f09188de0c4',
  'source': 'sketchfab'},
 {'sha256': '39ccad9118884c8e416005646d85d1f668c91c944adbc6187262a829ab2d14e3',
  'fileIdentifier': 'https://sketchfab.com/3d-models/70583f5df06448c1bba076df3afcdbbb',
  'source': 'sketchfab'},
 {'sha256': 'f28847e18692b047fc15047749257a770b2f1b7e03124eaf4912d7cbdf796316',
  'fileIdentifier': 'https://sketchfab.com/3d-models/44be138ae8e2409bbbca44a96fc67d45',
  'source': 'sketchfab'},
 {'sha256': '396405650da99fd6e

In [127]:
import multiprocessing

In [128]:
multiprocessing.cpu_count()

16

In [ ]:
!htop

2486999 dtrofimov 1144M  318M  97527  0.5  0:01.18 python3 main.pyon3 main.p[1;1Hsystemd/sM 93208 22004 S 1  0:06.61 /usr/bin/python46336.3G 13.7G 7399221.5  1:04.20 /vol/isy-rl/dtr46496.3G 13.7G 73992135 /vol/isy-rl/dtr23654699944M  312M  97521.1612634256  4744  3628 R 016 htoppy0   0  4816  1232   900 S  0.0  0.0  0:00.00 /usr/sbin/rpc.s920 _rpc       20   0  8100  1308   876 S  0.0  0.0  0:02.59 /sbin/rpcbind -921 systemd-o  20   0 14828  1664   864 S  0.0  0.0 28:40.43 /lib/systemd/sy931 root       20   0  313M  5032  3356 S  0.0  0.0  0:12.05 /usr/sbin/rpc.g932 root       20   0  313M  5032  3356 S  0.0  0.0  0:02.20 /usr/sbin/rpc.g1145 root       20   0  2808    92     0 S  0.0  0.0  0:00.25 /usr/sbin/acpid1148 messagebu  20   0 37920  3992  1268 S  0.0  0.0  0:05.98 @dbus-daemon --1154 root       20   0 82808  1192   812 S  0.0  0.0  1:18.55 /usr/sbin/irqba1158 root       20   0  258M  8100  4544 S  0.0  0.0  0:02.83 /usr/libexec/poF1Help  F2Setup F3SearchF4FilterF5Tree  F6SortB

  4445446264  1916   808 S  0.7  0.0  0:07.37 /usr/bin/pipewi0169M  3210.76969918M  307M  9736 S  0.753/dtr6990169M  307M  9752 S  0.746995065M  304M  9752 D  0.7  0.5  0:00.55 python3 main.py6997069M  308M  9752 D  0.7  01.13 python3 main.py06049M  308M  9689R 66700528670D  0.716993D  0.776997069M  308M  9752 S  0.71.097000065M  304M  9752 D  0.7  0.5  0:00.59 python3 main.py7001065M  304M  975 072 python3 main.py7002283|||3.3|7| 0.7|7|2.7||7 0|7|2.0||3.9|1.3130D  2.057069M  308D  2.0120D 635S 54699944M  330M  97521.2D 8D 946336.3G 13.7G 73992 S  0.7 21.5  1:04.25 /vol/isy-rl/dtr46496.3G 13.7G 73992 S  0.7 2139 /vol/isy-rl/dtr70015M  304D 0.73228643127M  329M  973666  0.0 ||1.3 0| 0 0||7| 0.7| 3  0.0377 3.28 1.1627003127M  3336 S  2.77025M  304M  9736 D  1.30.619144M  336S 1.27700228 D 660.7|7| 0 0|7|7 0||2.034040169M  312M  9752 S 641946336.3G 13.7G 73992 S  0.7 21.5  1:04.27 /vol/isy-rl/dtr46496.3G 13.7G 73992 S  0.7 210.40 /vol/isy-rl/dtr6998712634256  4744  3628 R 019 htop1695 roo

      1 root       164M  9544  41440  0.0  0:39.97 /lib/systemd/s1;1H2487004 dtrofimov 1069M  308M  96927  0.5  0:00.92 python3 main.py2487126 dtrofimov 34256  4744  3628 R  0.7 0:00.24 htop  164M  9544  414439.97 /lib/systemd/sy1;1H4;45HD  2.01.032065M  3042 S  2.0757004069M  308M  962.0  0.5  0:00.85 python3 main.py61.32852 D  1.3757000D  1.36770057328 S  1.38146336.3G 13.7G 739921.5  1:04.33 /vol/isy-rl/dtr69918M  307M  9736 S 069971069M  308M  9752 S 5  0:01.18 python3 main.py|74  0.0  0.0 0|1.3 0|3.3|7| 0246992065M  30457370005277188S  1.351.371.3746336.3G 13.7G 73992 S  0.7 21.5  1:04.34 /vol/isy-rl/dtr4646.3G 13.7G 73992 S  0.7 2110 /vol/isy-rl/dtr46496.3G 13.7G 73992 S  0.7 2143 /vol/isy-rl/dtr6990169M  328M  9752 S  0.773699065M  304M  975 0.5  0:00.90 python3 main.py47M  304M  97529755M  3040.71 00.7||2|1.3||1.3|0.7  0.0 0625700208049M  308M  9692 S  1.38946336.3G 13.7G 73992 S  0.7 21.5  1:04.35 /vol/isy-rl/dtr18M  307M  9736 D  0.70.616996 S  0.7469935M  304M  9752 S  0.791

2484633 dtrofimov  20   0 16.3G 13.7G 73992 S  0.7 21.5  1:04.46 /vol/isy-rl/dtr46496.3G 13.7G 739922150 /vol/isy-rl/dtr7  0.0 0  0.0||2.7 09752 D  2.0718M  307M  9736 S  10.7269935M  304M  9756 D  1.31.04[816996065M  304M  9752 D  1.3  0.5  0:01.17 python3 main.py70025M  3042 S  1.39470049M  308M  9692 D  1.30.9770057328 S  1.30.8946336.3G 13.7G 739921.5  1:04.43 /vol/isy-rl/dtr4646.3G 13.7G 739921.5  0:00.49 /vol/isy-rl/dtr699236 D 7906 S 8147:0170035M  304M  9736 D  1.3246336.3G 13.7G 73992 S  0.7 21.5  1:04.44 /vol/isy-rl/dtr0169M  3422 S  0.70.7618M  307M  9736 D  0.7738149M  317S  0.70.856999144M  379M  9752 S  0.7  0.6  0:01.4305M  304M  9756 S  0.78260.790712634256  4744  3628 R  0.7  0.0  0:00.26 htop|1.3||3.3||1.3|| 0| 228.16 3.50 1.302699952 R  4.0496998149M  321M  9752 S  2.0  0.5  0:00.88 python3 main.py7004008M  9692 D  1.39946336.3G 13.7G 73992 S  0.7 21.5  1:04.45 /vol/isy-rl/dtr069M  346777000065M  30465  0:00.8920 _rpc       20   0  8100  1308   876 S  0.0  0.0  0:02

2484633 dtrofimov 16.3G 13.7G 7399221.5  1:04.73 /vol/isy-rl/dtr490:00.65 /vol/isy-rl/dtr8D 99299101321M  356M 63196 S  0.7  0.6  0:02.52 python3 main.pm1065M  304M  9752 D  25  0:00.97 python3 main.py2487002 dtrofimov 1065M  304M  9732 D  25  0:01.01 python3 main.py[2;45H|2|7|1.3|7.3|7|1.3||1.3|7|1.3||1.3||3.3|1.3||3.33406995098M  307M  9756 S  4.0  0.5  0:00.89 python3 main.py69901065M  304M  9752 D  3.3  0.5  0:01.23 python3 main.py2487004 dtrofimov 1069M  308M  9692 D  3.3  0.5  0:01.06 python3 main.p|0.74.00.70.730.7  0.0 0.0| 0.7  0.00.7| 2.7|7211636 S  2136 S  2.01569905M  304M  9752 S  125699265M  304M  9736 S  10.884112M  312S  1.31512 S  1.3970035M  304S  1.31.1246336.3G 13.7G 73992 S  0.7 21.5  1:04.67 /vol/isy-rl/dtr6097M  3116 S  0.72579M  3080.733S  0.71232 S  0.71.02573M  312M  9728 S  0.72 0 1.4 0 0 0|1 0 0| 0 0|| 01.33 4.933700573M  312M  97281.0502 S  1.32718M  307M  97360.8379M  308M  9752 D 1.357000065M  3046 D 0.93231.044636.3G 13.7G 73992 S  0.7 21.5  1:04.68 /vol

...

2484649 dtrofimov  20   0 16.3G 13.7G 73992 S  0.7 21.5  0:02.07 /vol/isy-rl/dtr2486990 dtrofimov  20   0 1067M  306M  9756 D  0.7  0.5  0:04.02 python3 main.py24869991066M  306M  9756 D 5  0:04.25 python3 main.py[m 20   0 37920  3992  1268 S  0.0  0.0  0:05.98 @dbus-daemon --[m0  0:01.34 htopH4.036S 4.66700072M  311M  9756 S 3546336.3G 13.7G 73992 S  0.7 21.5  1:07.13 /vol/isy-rl/dtr46496.3G 13.7G 73992 S  0.7 212.01 /vol/isy-rl/dtr6992065M  304M  9740 D  0.7  0.5  0:02.98 python3 main.py94067M  306M  9820 D 5  0:03.3079M  308D 6.185M  304M  97564.097003113M  318S 3.76|0.7  0.0 0|  0|||1.3|7||3.3627002072M  311M  973274.078065M  3044.126999106M  333M  975625182M  3302.97 0|7||1.3||1.3 0| 0.7|| 2.67214 9.70 5.0336995182M  335603.009106M  335057002072M  311M  9732 R 108065M  3044.14||1.3 0| 0.7  0.0|7|||7157002072M  3132 D  3.34.15582M  3333.036999106M  338M  9756 S 08D 6|7  0.0||1.3|7||1.3| 0  0.05266995182M  337M  9756 R  4.03.090067M  3047984011 0|1.32.6 0||3.36121 9.76 5.07841S  5.3

      1 root       164M  9544  41440  0.0  0:39.98 /lib/systemd/sy7001071M  310M  9756 D  1.3  0.5  0:03.74 python3 main.py700371M  310M  9740 S  1.33.9646336.3G 13.7G 73992 S  0.7 21.5  1:07.41 /vol/isy-rl/dtr464916.3G 13.7G 73992 S  0.7 21.5  0:02.18 /vol/isy-rl/dtr2487000 dtrofimov 1072M  311M  9756 D  0.7  0.5  0:03.47 python3 main.pm69967M  3064.07 0|||2 0 0.0||2 0|7||3.3714 10.38 5.464700272M  311M  9732 D  2.022D 937247M  306M  98203.3|1.3|3.3 0  0|1.3|1.3 |7 02.0| 0.7| 0.7|755699769M  308M  9756 S  46.2555M  304M  9756 S  2.0702400565M  304M  97564.75|2.0|5|7|7 0.04.0|71.33.9|7|6 0||5.2 0718M  307M  9740 D  2.73.53874.2907M  306M  9756 D  2.04.105S  2.03.73700072M  3112.03.5053M  312M  9728 D  2.026699265M  304D 07699109M  306M  9756 S  1.3  0.5  0:03.74 python3 main.py6994067M  306M  9820 D  1.3  03.41 python3 main.py699966M  3061.34.3311M  310M  9756 D  1.33.76272M  311M  9732 D  1.34.2531M  310M  9740 D  1.39866.7||1.32 0|2.0|0.7 0 1.32.782.85 10.29 5.45879M  308M  97566.291

   1866 root      24876  2024   908 S 0  6:24.85 /usr/sbin/cups-46496.3G 13.7G 73992212.26 /vol/isy-rl/dtr18M  307M  97403.786995065M  30D 4.017560916997M  306M  9820058[417001139M  318R 3.927004069M  309M  9692724614 260M 93188 21984 S  0.7  0.1  0:06.97 /usr/bin/python46336.3G 13.7G 73992 S  0.7 21.5  1:07.49 /vol/isy-rl/dtr25M  304M  9740 S 3.14[156999066M  3064.40001129M  315M  9756 S 5  0:03.58 python3 main.py2487003 dtrofimov 1071M  310M  97407  0.5  0:04.05 python3 main.p1||1.3 0|2.0|7||7| 1.311.39 10.07 5.43918M  307M  9740 D  2.73.700067M  306D  1.34.263109M  326M  9756 S  1.39347M  306M  9820 S  1.33.603071M  310M  9740 D  1.34.0746336.3G 13.7G 7390.7 21.5  1:07.50 /vol/isy-rl/dtr4916.3G 13.7G 7399221.5  0:02.22 /vol/isy-rl/dtr6995065M  304M  9756 0.5  0:03.86 python3 main.py676564.888D 4.447000129M  313.59139M  3219322M  311M  9732342.72.7 0|7 01.3322025M  304S  1.316700371M  310M  9740097004069M  309M  969274700573M  312M  9728394636.3G 13.7G 73992 S  0.7 21.5  1:07.51 /v

    889 root       4816  1232   900 S  0.0  0.0  0:00.00 /usr/sbin/rpc.s    920 _rpc       8100  1308   876 S  0.02.59 /sbin/rpcbind -921 systemd-o14828  1664   8628:40.6093120   0  313M  5032  3356 0  0:12.05 /usr/sbin/rpc.g932 313M  5032  33562.20 /usr/sbin/rpc.gH26992106M  306M  9740 R  2.025700272M  311M  9732 D  2.04246336.3G 13.7G 73992 S  1.3 21.5  1:07.59 /vol/isy-rl/dtr18M  307M  9740 S  1.33.823109M  342M  9756 S 4.047820 S 3.73699565M  304M  9756 D 066999066M  306M  9756 S  1.3  0.5  0:04.57 python3 main.py7003071M  310M  9740 S  1.3  0.5  0:04.207009M  309M  9692 D  1.34.046496.3G 13.7G 73992 S  0.7 212.27 /vol/isy-rl/dtr699067M  306D 4.34029M  3293.66 06 0| 0.7|7 0 0 02.06157 9.95 5.47 48S8699467M  306M  9820 S  1.33.756996076M  310M  9756 0.5  0:04.96 python3 main.py46336.3G 13.7G 73992 S  0.7 21.5  1:07.60 /vol/isy-rl/dtr40.7185M  304M  9756 D  0.74.597001139M  339S  0.72 0|71.3| 0|7||1.352510R  3.3337000129M  334M  9756687001139M  34204[1146496.3G 13.7G 73992212.28 /vo

    931 root       313M  5032  3356 S  0.0  0.0  0:12.05 /usr/sbin/rpc.g    932 root       313M  5032  3356 S  0.0  0.0  0:02.20 /usr/sbin/rpc.g 4816  1232   9000  0.0  0:00.00 /usr/sbin/rpc.s    920 _rpc       8100  1308   8760  0.059 /sbin/rpcbind -    921 systemd-o 14828  1664   864 S  0.0  0.0 28:40.60 /lib/systemd/s24876  2024   9080  6:24.86 /usr/sbin/cups-46336.3G 13.7G 7399221.5  1:07.68 /vol/isy-rl/dtr46496.3G 13.7G 73992 S  0.7 212.33 /vol/isy-rl/dtr2486993 dtrofimov 1065M  305M  9756 D  0.7  0.5  0:04.09 python3 main.p
2487126 dtrofimov  20   0 34256  4388  3272 R  0.7  0.0  0:01.56 htop   920 _rpc      8100  1308   8760  0.0  0:02.59 /sbin/rpcbind -    921 systemd-o 14828  1664   8640  0.0 28:40.60 /lib/systemd/sy    931 root       313M  5032  33560  0.0  0:12.05 /usr/sbin/rpc.g    932 root       313M  5032  3356 S  0.0  0.0  0:02.20 /usr/sbin/rpc.g   1145 root       2808    92     0 S  0.0  0.0  0:00.25 /usr/sbin/acpid   1148 messagebu 37920  3992  1268 S  0.0  0.0  0:05.9

In [133]:
objects_to_download

[{'sha256': '079c1fbedddddc5b9462a22ec9e043e808aac72c0b138b2173b4f191971135c6',
  'fileIdentifier': 'https://sketchfab.com/3d-models/bc0fd04422374b3987162a6c499280d8',
  'source': 'sketchfab'},
 {'sha256': '8a784c4a068239e24666b84acd05083ea7cb5092e3c0bc7550c2e13e5798786f',
  'fileIdentifier': 'https://sketchfab.com/3d-models/2c3379f8126743c0b6e564c70ff859bc',
  'source': 'sketchfab'},
 {'sha256': '4e41726df0b2eb46fa66be100b6c9389c2fb215fa87e17fc818cbea1d920c34d',
  'fileIdentifier': 'https://sketchfab.com/3d-models/007ab9e6f1c047b287bc0f09188de0c4',
  'source': 'sketchfab'},
 {'sha256': '39ccad9118884c8e416005646d85d1f668c91c944adbc6187262a829ab2d14e3',
  'fileIdentifier': 'https://sketchfab.com/3d-models/70583f5df06448c1bba076df3afcdbbb',
  'source': 'sketchfab'},
 {'sha256': 'f28847e18692b047fc15047749257a770b2f1b7e03124eaf4912d7cbdf796316',
  'fileIdentifier': 'https://sketchfab.com/3d-models/44be138ae8e2409bbbca44a96fc67d45',
  'source': 'sketchfab'},
 {'sha256': '396405650da99fd6e

In [136]:
sampled_json

{'3a8400337ce34c728a23fe65660e76a1': {'uri': 'https://api.sketchfab.com/v3/models/3a8400337ce34c728a23fe65660e76a1',
  'uid': '3a8400337ce34c728a23fe65660e76a1',
  'name': 'Chair',
  'staffpickedAt': None,
  'viewCount': 9,
  'likeCount': 0,
  'animationCount': 0,
  'viewerUrl': 'https://sketchfab.com/3d-models/3a8400337ce34c728a23fe65660e76a1',
  'embedUrl': 'https://sketchfab.com/models/3a8400337ce34c728a23fe65660e76a1/embed',
  'commentCount': 0,
  'isDownloadable': True,
  'publishedAt': '2019-05-03T19:46:02.400841',
  'tags': [{'name': 'c',
    'slug': 'c',
    'uri': 'https://api.sketchfab.com/v3/tags/c'},
   {'name': 'furniture',
    'slug': 'furniture',
    'uri': 'https://api.sketchfab.com/v3/tags/furniture'},
   {'name': 'chair',
    'slug': 'chair',
    'uri': 'https://api.sketchfab.com/v3/tags/chair'},
   {'name': 'model',
    'slug': 'model',
    'uri': 'https://api.sketchfab.com/v3/tags/model'}],
  'categories': [],
  'thumbnails': {'images': [{'uid': 'f813a03e220f47f39ff

In [54]:
classes = list(chain(*list(objects_to_filter.values())))
items = sampled_json.items()

for name, properties in items:
    for class_name in classes:
        if class_name in properties["name"].lower():
            sampled_json[name]["class"] = class_name
            break

In [ ]:
with open("sampled_classes_january.json", "w") as f:
    json.dump(sampled_json, f)

In [53]:
sampled_json

44478

In [ ]:
sampled_json.popitem()[1][]